# 🔬 Karachi AQI Predictor — Feature Analysis

This notebook analyzes:
1. Autocorrelation — how well past AQI predicts future AQI
2. Feature-target correlations per horizon
3. Curse of dimensionality — optimal feature count per horizon
4. Delta vs Direct prediction comparison
5. Weather feature impact on AQI


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_squared_error
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': '#0f172a', 'axes.facecolor': '#1e293b',
    'axes.edgecolor': '#334155', 'axes.labelcolor': '#94a3b8',
    'xtick.color': '#64748b', 'ytick.color': '#64748b',
    'text.color': '#f1f5f9', 'grid.color': '#334155', 'grid.alpha': 0.5,
})

df = pd.read_parquet('../data/interim/aqi_features_cleaned.parquet')
df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df.sort_values('timestamp').reset_index(drop=True)
print(f'Loaded: {len(df):,} rows × {len(df.columns)} cols')
print(f'Period: {df.timestamp.min().date()} → {df.timestamp.max().date()}')

## 1. AQI Autocorrelation Analysis

In [ ]:
# Compute autocorrelation at various lags
lags = [1, 2, 3, 6, 12, 24, 48, 72, 168]
autocorrs = [df['aqi'].autocorr(lag=l) for l in lags]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('AQI Autocorrelation Analysis', fontsize=13, color='#f1f5f9', fontweight='bold')

# Bar chart
colors = ['#22c55e' if a > 0.7 else '#eab308' if a > 0.4 else '#ef4444' for a in autocorrs]
bars = axes[0].bar([str(l)+'h' for l in lags], autocorrs, color=colors, alpha=0.85)
axes[0].axhline(0.7, color='#22c55e', linestyle='--', lw=1.5, alpha=0.7, label='Strong (0.7)')
axes[0].axhline(0.4, color='#eab308', linestyle='--', lw=1.5, alpha=0.7, label='Moderate (0.4)')
axes[0].set_title('Autocorrelation at Key Lags', color='#94a3b8')
axes[0].set_xlabel('Lag')
axes[0].set_ylabel('Pearson Correlation')
axes[0].legend()
for bar, val in zip(bars, autocorrs):
    axes[0].text(bar.get_x() + bar.get_width()/2, val + 0.01,
                 f'{val:.2f}', ha='center', va='bottom', fontsize=8)

# Scatter: aqi[t] vs aqi[t+24]
sample_size = min(1500, len(df))
idx = np.random.choice(len(df)-24, sample_size, replace=False)
x_vals = df['aqi'].iloc[idx].values
y_vals = df['aqi'].iloc[idx + 24].values
sc = axes[1].scatter(x_vals, y_vals, c=x_vals, cmap='YlOrRd', alpha=0.4, s=8)
plt.colorbar(sc, ax=axes[1], label='Current AQI')
z = np.polyfit(x_vals, y_vals, 1)
axes[1].plot(sorted(x_vals), np.poly1d(z)(sorted(x_vals)), 'w--', lw=2,
             label=f'r = {np.corrcoef(x_vals, y_vals)[0,1]:.3f}')
axes[1].plot([x_vals.min(), x_vals.max()], [x_vals.min(), x_vals.max()],
             '#94a3b8', lw=1, alpha=0.5, label='Perfect prediction')
axes[1].set_xlabel('AQI at time t')
axes[1].set_ylabel('AQI at time t+24h')
axes[1].set_title('AQI[t] vs AQI[t+24h]', color='#94a3b8')
axes[1].legend()

plt.tight_layout()
plt.savefig('autocorrelation.png', dpi=150, bbox_inches='tight', facecolor='#0f172a')
plt.show()

print('\nAutocorrelation summary:')
for l, r in zip(lags, autocorrs):
    strength = 'Strong' if r > 0.7 else 'Moderate' if r > 0.4 else 'Weak'
    print(f'  Lag {l:3d}h: r={r:.3f}  ({strength})')

## 2. Feature-Target Correlation per Horizon

In [ ]:
horizons = [1, 6, 12, 24, 48, 72]
exclude = [c for c in df.columns if 'aqi_t_plus' in c or 'aqi_delta' in c
           or c in ['timestamp', 'city']]
feature_cols = [c for c in df.columns if c not in exclude]

# Top features per horizon
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Top 15 Features by Correlation with AQI Delta per Horizon',
             fontsize=13, color='#f1f5f9', fontweight='bold')
axes = axes.flatten()

for i, h in enumerate(horizons):
    target = f'aqi_delta_{h}h'
    if target not in df.columns:
        continue
    df_h = df.dropna(subset=[target])
    corrs = df_h[feature_cols].corrwith(df_h[target]).abs().dropna()
    top15 = corrs.nlargest(15)

    colors = ['#38bdf8' if 'aqi' in c else '#34d399' if 'pm' in c
              else '#a78bfa' if any(w in c for w in ['temp','wind','humid','press','cloud','precip'])
              else '#fbbf24' for c in top15.index]

    axes[i].barh(range(len(top15)), top15.values, color=colors, alpha=0.85)
    axes[i].set_yticks(range(len(top15)))
    axes[i].set_yticklabels([c.replace('_', ' ') for c in top15.index], fontsize=8)
    axes[i].set_title(f'+{h}h Horizon', color='#94a3b8')
    axes[i].set_xlabel('|Correlation|')
    axes[i].invert_yaxis()

# Legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#38bdf8', label='AQI features'),
    Patch(facecolor='#34d399', label='PM features'),
    Patch(facecolor='#a78bfa', label='Weather features'),
    Patch(facecolor='#fbbf24', label='Other'),
]
fig.legend(handles=legend_elements, loc='lower center', ncol=4,
           bbox_to_anchor=(0.5, -0.02), framealpha=0.3)

plt.tight_layout()
plt.savefig('feature_correlations.png', dpi=150, bbox_inches='tight', facecolor='#0f172a')
plt.show()

## 3. Curse of Dimensionality — Optimal Feature Count

In [ ]:
feature_counts = [5, 10, 15, 20, 30, 50, 70, 96]
results = {h: [] for h in horizons}

print('Testing feature counts...')
for h in horizons:
    target = f'aqi_delta_{h}h'
    if target not in df.columns:
        continue
    df_h = df.dropna(subset=[target])
    corrs = df_h[feature_cols].corrwith(df_h[target]).abs().dropna().sort_values(ascending=False)
    n = len(df_h)
    split = int(n * 0.8)

    for k in feature_counts:
        top_k = corrs.head(min(k, len(corrs))).index.tolist()
        X = df_h[top_k].fillna(df_h[top_k].mean())
        y = df_h[target]
        model = Pipeline([('sc', StandardScaler()), ('r', Ridge(100))])
        model.fit(X.iloc[:split], y.iloc[:split])
        r2 = r2_score(y.iloc[split:], model.predict(X.iloc[split:]))
        results[h].append(r2)
    print(f'  +{h}h done')

print('Done!')

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('R² vs Number of Features per Horizon\n(Curse of Dimensionality Analysis)',
             fontsize=13, color='#f1f5f9', fontweight='bold')
axes = axes.flatten()

for i, h in enumerate(horizons):
    r2_vals = results[h]
    if not r2_vals:
        continue
    ax = axes[i]
    ax.plot(feature_counts[:len(r2_vals)], r2_vals, 'o-', color='#38bdf8', lw=2, ms=8)
    best_idx = np.argmax(r2_vals)
    ax.scatter(feature_counts[best_idx], r2_vals[best_idx],
               color='#fbbf24', s=150, zorder=5, label=f'Best: top{feature_counts[best_idx]} R²={r2_vals[best_idx]:.3f}')
    ax.axhline(0, color='#ef4444', linestyle='--', lw=1, alpha=0.7, label='R²=0 (random baseline)')
    ax.set_title(f'+{h}h Horizon', color='#94a3b8')
    ax.set_xlabel('Number of Features')
    ax.set_ylabel('Test R²')
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('curse_of_dimensionality.png', dpi=150, bbox_inches='tight', facecolor='#0f172a')
plt.show()

print('\nOptimal feature count per horizon:')
for h in horizons:
    if results[h]:
        best_idx = np.argmax(results[h])
        print(f'  +{h}h: top {feature_counts[best_idx]} features  (R²={results[h][best_idx]:.3f})')

## 4. Delta vs Direct Prediction Comparison

In [ ]:
print('Comparing Delta framing vs Direct prediction...')
comparison = []

base_features = [c for c in feature_cols if c in df.columns][:50]

for h in horizons:
    delta_col = f'aqi_delta_{h}h'
    abs_col = f'aqi_t_plus_{h}h'
    if delta_col not in df.columns or abs_col not in df.columns:
        continue

    df_h = df.dropna(subset=[delta_col, abs_col])
    available_feats = [c for c in base_features if c in df_h.columns]
    X = df_h[available_feats].fillna(df_h[available_feats].mean())
    n = len(df_h)
    split = int(n * 0.8)

    # Direct prediction
    y_direct = df_h[abs_col]
    m_direct = Pipeline([('sc', StandardScaler()), ('r', Ridge(100))])
    m_direct.fit(X.iloc[:split], y_direct.iloc[:split])
    preds_direct = m_direct.predict(X.iloc[split:])
    r2_direct = r2_score(y_direct.iloc[split:], preds_direct)

    # Delta prediction
    y_delta = df_h[delta_col]
    m_delta = Pipeline([('sc', StandardScaler()), ('r', Ridge(100))])
    m_delta.fit(X.iloc[:split], y_delta.iloc[:split])
    preds_delta = m_delta.predict(X.iloc[split:])
    # Reconstruct absolute
    current_aqi = df_h['aqi'].iloc[split:].values
    preds_reconstructed = current_aqi + preds_delta
    r2_delta = r2_score(y_direct.iloc[split:], preds_reconstructed)

    comparison.append({'Horizon': f'+{h}h', 'Direct R²': r2_direct, 'Delta R²': r2_delta,
                       'Improvement': r2_delta - r2_direct})
    print(f'  +{h}h: Direct={r2_direct:.3f}  Delta={r2_delta:.3f}  Δ={r2_delta-r2_direct:+.3f}')

comp_df = pd.DataFrame(comparison)
print('\n', comp_df.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

x = np.arange(len(comp_df))
w = 0.35
bars1 = ax.bar(x - w/2, comp_df['Direct R²'], w, label='Direct Prediction', color='#ef4444', alpha=0.8)
bars2 = ax.bar(x + w/2, comp_df['Delta R²'],  w, label='Delta + Reconstruct', color='#22c55e', alpha=0.8)

ax.axhline(0, color='white', lw=0.8, linestyle='--', alpha=0.5)
ax.set_xticks(x)
ax.set_xticklabels(comp_df['Horizon'])
ax.set_ylabel('Test R²')
ax.set_title('Delta Framing vs Direct Prediction — Ridge Regression', color='#f1f5f9', fontsize=13)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

for bar in bars1:
    h_val = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h_val + 0.01 if h_val >= 0 else h_val - 0.05,
            f'{h_val:.2f}', ha='center', va='bottom', fontsize=8, color='#fca5a5')
for bar in bars2:
    h_val = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h_val + 0.01 if h_val >= 0 else h_val - 0.05,
            f'{h_val:.2f}', ha='center', va='bottom', fontsize=8, color='#86efac')

plt.tight_layout()
plt.savefig('delta_vs_direct.png', dpi=150, bbox_inches='tight', facecolor='#0f172a')
plt.show()

## 5. Weather Impact on AQI

In [ ]:
weather_cols = ['temperature', 'humidity', 'wind_speed', 'pressure', 'precipitation', 'cloud_cover']
available_weather = [c for c in weather_cols if c in df.columns]

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('AQI vs Weather Features — Karachi', fontsize=13, color='#f1f5f9', fontweight='bold')
axes = axes.flatten()

colors_map = {'temperature': '#ef4444', 'humidity': '#38bdf8', 'wind_speed': '#a78bfa',
              'pressure': '#fbbf24', 'precipitation': '#34d399', 'cloud_cover': '#94a3b8'}

for i, col in enumerate(available_weather[:6]):
    ax = axes[i]
    sample = df[['aqi', col]].dropna().sample(min(1500, len(df)))
    r = sample['aqi'].corr(sample[col])
    color = colors_map.get(col, '#38bdf8')
    ax.scatter(sample[col], sample['aqi'], c=color, alpha=0.3, s=6)
    # Trend
    z = np.polyfit(sample[col], sample['aqi'], 1)
    x_line = np.linspace(sample[col].min(), sample[col].max(), 100)
    ax.plot(x_line, np.poly1d(z)(x_line), 'w-', lw=2)
    ax.set_xlabel(col.replace('_', ' ').title())
    ax.set_ylabel('AQI')
    ax.set_title(f'{col.replace("_", " ").title()}  (r={r:+.3f})', color='#94a3b8')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('weather_impact.png', dpi=150, bbox_inches='tight', facecolor='#0f172a')
plt.show()

print('\nWeather feature correlations with AQI:')
for col in available_weather:
    r = df['aqi'].corr(df[col])
    direction = '↑ higher AQI' if r > 0 else '↓ lower AQI'
    print(f'  {col:20s}: r={r:+.3f}  →  {direction}')

## Summary of Key Findings

| Finding | Detail |
|---|---|
| **Autocorrelation** | AQI is strongly autocorrelated at 1h (r≈0.99), moderate at 24h (r≈0.6), weak at 72h |
| **Best predictor for short horizons** | `aqi_lag_1h`, `aqi_rolling_3h` — recent AQI dominates |
| **Best predictor for long horizons** | PM2.5 lags, wind speed — pollutant persistence matters |
| **Curse of dimensionality** | 48h/72h models benefit from 10-30 features, not 96 |
| **Delta framing** | Consistently outperforms direct prediction across all horizons |
| **Weather impact** | Wind speed most negatively correlated (disperses pollutants) |

### Future Work
- SHAP-based automatic feature selection per horizon
- Weather forecast features (not just current weather) for 48h/72h
- LSTM/Transformer for sequential pattern capture
